# Train and register diabets model

This notebook trains a `RandomForestRegressor` on the sklearn diabetes dataset, logs the run to MLflow, registers the model as `diabets`, and downloads the selected model version for the service.

In [ ]:
from pathlib import Path

import mlflow
import mlflow.sklearn
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
MLFLOW_TRACKING_URI = 'http://host.docker.internal:5001'
EXPERIMENT_NAME = 'hw24-diabetes'
REGISTERED_MODEL_NAME = 'diabets'
LOCAL_MODEL_DIR = Path('../model').resolve()

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
dataset = load_diabetes(scaled=False)
X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('model', RandomForestRegressor(n_estimators=300, random_state=42))
    ]
)

In [ ]:
with mlflow.start_run(run_name='random_forest_diabets') as run:
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)

    mlflow.log_metric('mae', mae)
    mlflow.log_param('n_estimators', 300)
    mlflow.log_param('random_state', 42)

    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path='model',
        registered_model_name=REGISTERED_MODEL_NAME,
    )

run.info.run_id, mae, model_info.model_uri

In [ ]:
client = mlflow.tracking.MlflowClient()
latest = client.get_latest_versions(REGISTERED_MODEL_NAME, stages=['None'])[-1]
model_uri = f'models:/{REGISTERED_MODEL_NAME}/{latest.version}'
LOCAL_MODEL_DIR

In [ ]:
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
loaded_model = mlflow.sklearn.load_model(model_uri)
mlflow.sklearn.save_model(loaded_model, str(LOCAL_MODEL_DIR))
latest.version